# Week 4 — Pruning (Colab)

**런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 켜고, 위에서부터 셀을 순서대로 실행하세요.

결과: Google Drive `skkai-jetson/` 폴더에 양자화된 `.gguf` 하나. 노트북으로 받아서 Jetson에 옮기면 됩니다.

| 담당 | 설정 |
|---|---|
| A | `MODE="layers"`, `N_DROP=2` |
| B | `MODE="layers"`, `N_DROP=4` |
| C | `MODE="layers"`, `N_DROP=6` |
| D | `MODE="layers"`, `N_DROP=8` |
| E | `MODE="magnitude"` |

In [ ]:
# ===== 여기만 바꾸세요 =====
MODE   = "layers"      # "layers" (레이어 제거) 또는 "magnitude" (가중치 50%를 0으로)
N_DROP = 4             # MODE="layers"일 때 뺄 층 수 (1.5B는 전체 28층)
QUANT  = "Q4_K_M"      # Week 3에서 확정한 포맷 (imatrix가 필요한 IQ 계열은 여기서 불가)
SAVE_F16 = False       # True면 F16 GGUF도 Drive에 저장 (약 3GB, Jetson에서 imatrix 만들 때 필요)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
NAME = f"qwen1p5b-drop{N_DROP}" if MODE == "layers" else "qwen1p5b-mag50"
print(NAME)

In [ ]:
# (1~3분) GPU 확인, 패키지, llama.cpp(변환 스크립트 + llama-quantize), Google Drive 연결
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q sentencepiece protobuf datasets peft trl
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null && echo "llama-quantize ready"

from google.colab import drive
drive.mount("/content/drive")
import os
OUT_DIR = "/content/drive/MyDrive/skkai-jetson"
os.makedirs(OUT_DIR, exist_ok=True)
print("results ->", OUT_DIR)

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from huggingface_hub import snapshot_download

def load_base(model_id):
    path = snapshot_download(model_id, local_dir="/content/base")
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map="cuda")
    return tok, model

def drop_layers(tok, model, n_drop):
    """ShortGPT 방식: 입력과 출력이 비슷한 레이어(하는 일이 적은 레이어)를 제거"""
    if n_drop == 0:
        return model
    text = "\n".join(t for t in load_dataset("wikitext", "wikitext-2-raw-v1", split="train")["text"] if t.strip())
    ids = tok(text[:400000], return_tensors="pt").input_ids[:, :32 * 512].view(32, 512).cuda()
    L = model.config.num_hidden_layers
    sim = torch.zeros(L)
    with torch.no_grad():
        for b in ids.split(4):
            hs = model(b, output_hidden_states=True).hidden_states
            for i in range(L):
                sim[i] += torch.nn.functional.cosine_similarity(hs[i].float(), hs[i + 1].float(), dim=-1).mean().cpu()
    sim /= 8
    print("layer similarity (높을수록 제거 후보):", [round(s, 3) for s in sim.tolist()])
    sim[0] = sim[-1] = -1e9  # 첫·마지막 레이어는 남김
    drop = sorted(sim.topk(n_drop).indices.tolist())
    print("drop layers:", drop)
    keep = [l for i, l in enumerate(model.model.layers) if i not in drop]
    model.model.layers = torch.nn.ModuleList(keep)
    for i, l in enumerate(model.model.layers):
        l.self_attn.layer_idx = i
    if getattr(model.config, "layer_types", None):
        model.config.layer_types = [t for i, t in enumerate(model.config.layer_types) if i not in drop]
    model.config.num_hidden_layers = len(keep)
    print(f"{L} -> {len(keep)} layers")
    return model

def magnitude50(model):
    """비정형 pruning: 행마다 절댓값이 작은 절반을 0으로"""
    with torch.no_grad():
        for layer in model.model.layers:
            for m in layer.modules():
                if isinstance(m, torch.nn.Linear):
                    w = m.weight
                    w.scatter_(1, w.abs().argsort(dim=1)[:, : w.shape[1] // 2], 0)
    return model

In [ ]:
# (1~2분) 원본 받기 + pruning
tok, model = load_base(MODEL_ID)
model = drop_layers(tok, model, N_DROP) if MODE == "layers" else magnitude50(model)

In [ ]:
# HF 폴더 -> F16 GGUF -> 양자화 GGUF -> Google Drive
import shutil, subprocess
hf_dir = f"/content/{NAME}"
model.save_pretrained(hf_dir); tok.save_pretrained(hf_dir)
del model; gc.collect(); torch.cuda.empty_cache()

f16 = f"/content/{NAME}-f16.gguf"
q = f"/content/{NAME}-{QUANT.lower()}.gguf"
subprocess.run(["python", "/content/llama.cpp/convert_hf_to_gguf.py", hf_dir, "--outfile", f16, "--outtype", "f16"], check=True)
subprocess.run(["/content/llama.cpp/build/bin/llama-quantize", f16, q, QUANT], check=True)

shutil.copy(q, OUT_DIR)
if SAVE_F16:
    shutil.copy(f16, OUT_DIR)
!ls -lh {OUT_DIR}
print("Drive의 skkai-jetson 폴더에서 노트북으로 내려받은 뒤 Jetson으로 scp 하세요.")